In [47]:
import pandas as pd
from pathlib import Path

q1 = pd.read_csv("../data/raw/Q1_2021.csv")
q2 = pd.read_csv("../data/raw/Q2_2021.csv")
directorio = pd.read_csv("../data/raw/Directorio.csv", encoding="latin-1")

In [48]:
directorio = directorio.drop(columns=["Unnamed: 2"], errors="ignore")
print("Directorio:", directorio.shape)
print(directorio.head(10))

Directorio: (30, 4)
   Client  Branch      Nombre        Estado
0    1520       1  Sucursal A       Yucatán
1    1520       2  Sucursal B       Yucatán
2    1520       3  Sucursal C       Yucatán
3    1520       5  Sucursal D       Yucatán
4    1521       6  Sucursal E       Yucatán
5    1521       7  Sucursal F  Quintana Roo
6    1521       8  Sucursal G  Quintana Roo
7    1521       9  Sucursal H  Quintana Roo
8    1521      10  Sucursal I  Quintana Roo
9    1521      11  Sucursal J  Quintana Roo


In [49]:
columnas_fecha = ["FechaFactura", "FechaApertura", "FechaEntrega"]

columnas_moneda = [
    "Venta$", "Costo$", "Utilidad$",
    "VentaMO$", "DescuentoMO$", "CostoMO$",
    "VentaMateriales$", "DescuentoMateriales$", "CostoMateriales$",
    "VentaTOT$", "DescuentoTOT$", "CostoTOT$",
    "VentaPartes$", "DescuentoPartes$", "CostoPartes$",
    "VentaTotal$", "CostoTotal$",
]

columnas_entero = [
    "IdRegistro", "Client", "Branch", "Factura",
    "NumeroOT", "VentasNetas", "Odometro", "Ano",
]

columnas_decimal = ["Margen", "Dias"]


def homologar_columnas(df):
    df = df.copy()

    # Fechas: formato de día/mes/año
    for columna in columnas_fecha:
        if columna in df.columns:
            df[columna] = pd.to_datetime(
                df[columna],
                dayfirst=True,
                errors="coerce"
            )

    
    for columna in columnas_moneda:
        if columna in df.columns:
            valores = (
                df[columna]
                .astype("string")
                .str.replace(r"[$,\s]", "", regex=True)
            )
            df[columna] = pd.to_numeric(valores, errors="coerce")


    for columna in columnas_entero:
        if columna in df.columns:
            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            ).astype("Int64")

 
    for columna in columnas_decimal:
        if columna in df.columns:
            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            ).astype("Float64")

   
    columnas_convertidas = (
        columnas_fecha
        + columnas_moneda
        + columnas_entero
        + columnas_decimal
    )
    for columna in df.columns:
        if columna not in columnas_convertidas:
            df[columna] = df[columna].astype("string")

    return df



q1 = homologar_columnas(q1)
q2 = homologar_columnas(q2)


In [50]:
q1["Fuente"] = "Q1"
q2["Fuente"] = "Q2"


datos = pd.concat([q1, q2], ignore_index=True)

columnas_vacias = datos.columns[datos.isna().all()]
datos = datos.drop(columns=columnas_vacias)

print(datos.head(10))
print("Columnas eliminadas por estar completamente vacías:", list(columnas_vacias))
print("Tamaño del conjunto unido:", datos.shape)
print("\nTipos de datos:")
print(datos.dtypes)

  FechaFactura  IdRegistro  Client  Branch FechaApertura FechaEntrega  \
0   2021-03-31      995439    1522      20    2021-03-09   2021-03-30   
1   2021-03-31      995440    1522      20    2021-03-09   2021-03-30   
2   2021-03-31      571394    1521      11           NaT   2021-03-01   
3   2021-03-31      995441    1522      20    2021-03-09   2021-04-01   
4   2021-03-31      995442    1522      20    2021-03-13   2021-04-14   
5   2021-03-31      995443    1522      20    2021-03-16   2021-03-31   
6   2021-03-31      995444    1522      20    2021-03-18   2021-04-18   
7   2021-03-31      995445    1522      20    2021-03-20   2021-04-20   
8   2021-03-31      995446    1522      20    2021-03-22   2021-04-10   
9   2021-03-31      995493    1522      20    2021-03-31   2021-03-31   

   Factura                 Taller    TipoOrden TipoPago  ...  CostoPartes$  \
0   114208  HOJALATERIA Y PINTURA      PUBLICO  CONTADO  ...       3223.98   
1   114227  HOJALATERIA Y PINTURA  ASEGU

In [51]:
referencia = directorio[
    ["Client", "Branch", "Nombre", "Estado"]
].drop_duplicates()

datos = datos.merge(
    referencia,
    on=["Client", "Branch"],
    how="left",
    validate="many_to_one"
)

datos[["Nombre", "Estado"]] = datos[["Nombre", "Estado"]].fillna("Sin Definir")

In [52]:
conteo_sin_definir = datos[["Nombre", "Estado"]].eq("Sin Definir").sum()
print(conteo_sin_definir)

Nombre    0
Estado    0
dtype: int64


In [53]:
datos[["Fuente", "IdRegistro", "Client", "Branch","Nombre","Estado"]].head(10)

,Fuente,IdRegistro,Client,Branch,Nombre,Estado
0,Q1,995439,1522,20,Sucursal O,Tabasco
1,Q1,995440,1522,20,Sucursal O,Tabasco
2,Q1,571394,1521,11,Sucursal J,Quintana Roo
3,Q1,995441,1522,20,Sucursal O,Tabasco
4,Q1,995442,1522,20,Sucursal O,Tabasco
5,Q1,995443,1522,20,Sucursal O,Tabasco
6,Q1,995444,1522,20,Sucursal O,Tabasco
7,Q1,995445,1522,20,Sucursal O,Tabasco
8,Q1,995446,1522,20,Sucursal O,Tabasco
9,Q1,995493,1522,20,Sucursal O,Tabasco


In [54]:
datos.describe(include="number")

,IdRegistro,Client,Branch,Factura,NumeroOT,Venta$,Costo$,Utilidad$,Margen,VentasNetas,...,DescuentoTOT$,CostoTOT$,VentaPartes$,DescuentoPartes$,CostoPartes$,VentaTotal$,CostoTotal$,Odometro,Ano,Dias
count,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,...,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,120258.0,107629.0
mean,807728.475153,1521.114604,17.25595,88072.328444,36800.306275,1509.822201,354.525347,1148.641459,0.263405,0.779266,...,0.0971,239.247808,2205.296406,29.908397,1472.029735,3685.21021,1826.555082,40747.949691,1777.690557,8.276041
std,408508.65948,0.728915,10.006845,99756.136714,14064.804573,2899.11403,1244.003522,2009.709113,95.020605,0.454098,...,8.221891,1169.631449,8760.394887,684.157405,6236.641603,10147.422259,6739.537505,71557.053914,653.454467,105.305796
min,66982.0,1520.0,1.0,1.0,10001.0,-66405.73,-71596.0,-42236.59,-30955.0,-1.0,...,-64.0,-71596.0,-269648.34,-38920.0,-230127.27,-269648.34,-230127.27,0.0,0.0,0.0
25%,473988.25,1521.0,9.0,20240.0,25126.25,173.3625,17.43,106.13,0.5,1.0,...,0.0,0.0,0.0,0.0,0.0,375.0,80.915,8331.0,2015.0,0.0
50%,847069.5,1521.0,18.0,50042.5,35605.5,772.2,75.52,645.31,0.83,1.0,...,0.0,0.0,813.995,0.0,498.1,1928.44,723.235,27775.0,2018.0,1.0
75%,1166583.75,1522.0,23.0,118060.75,46517.0,1837.115,306.8225,1463.655,0.91,1.0,...,0.0,26.05,1872.31,0.0,1120.605,3793.1,1555.275,56124.75,2020.0,5.0
max,1421942.0,1522.0,38.0,336611.0,65000.0,293165.94,120839.98,172325.96,1.05,1.0,...,2097.0,118000.0,449927.29,89627.0,354821.28,449927.29,356465.52,9808796.0,2218.0,32767.0


In [55]:
duplicados = datos["IdRegistro"].duplicated().sum()
nulos = datos["IdRegistro"].isna().sum()

print(f"Duplicados en IdRegistro: {duplicados}")
print(f"Nulos en IdRegistro: {nulos}")

Duplicados en IdRegistro: 0
Nulos en IdRegistro: 0


In [56]:
resumen_datos = pd.DataFrame({
    "Tipo de dato": datos.dtypes,
    "Valores faltantes": datos.isnull().sum(),
    "% Faltantes": (datos.isnull().sum() / len(datos) * 100).round(2)
})

resumen_datos

,Tipo de dato,Valores faltantes,% Faltantes
FechaFactura,datetime64[us],0,0.00
IdRegistro,Int64,0,0.00
Client,Int64,0,0.00
Branch,Int64,0,0.00
FechaApertura,datetime64[us],13386,11.13
FechaEntrega,datetime64[us],5919,4.92
Factura,Int64,0,0.00
Taller,string,0,0.00
TipoOrden,string,430,0.36
TipoPago,string,3800,3.16


In [57]:
columnas_criticas = ["Vin", "FechaApertura", "FechaEntrega"]

faltantes = datos[columnas_criticas].isna() | (
    datos[columnas_criticas]
    .astype("string")
    .apply(lambda columna: columna.str.strip().eq(""))
    .fillna(False)
)

datos["CamposCriticosFaltantes"] = faltantes.sum(axis=1)

datos["ClasificacionRecuperacion"] = "Completa"

datos.loc[
    faltantes["FechaEntrega"],
    "ClasificacionRecuperacion"
] = "Recuperable: falta FechaEntrega"

datos.loc[
    faltantes["FechaApertura"],
    "ClasificacionRecuperacion"
] = "Recuperable: falta FechaApertura"

datos.loc[
    faltantes["Vin"] | (datos["CamposCriticosFaltantes"] >= 2),
    "ClasificacionRecuperacion"
] = "No recuperable"


resumen_clasificacion = (
    datos["ClasificacionRecuperacion"]
    .value_counts()
    .rename_axis("Clasificacion")
    .reset_index(name="Entradas")
)

resumen_clasificacion["Porcentaje"] = (
    resumen_clasificacion["Entradas"] / len(datos) * 100
).round(2)

print(resumen_clasificacion)

                      Clasificacion  Entradas  Porcentaje
0                          Completa    103574       86.13
1  Recuperable: falta FechaApertura     10686        8.89
2   Recuperable: falta FechaEntrega      3240        2.69
3                    No recuperable      2758        2.29


In [58]:
carpeta_salida = Path("../data/preprocessed")
carpeta_salida.mkdir(parents=True, exist_ok=True)

ruta_salida = carpeta_salida / "datos_2021.csv"
datos.to_csv(ruta_salida, index=False, encoding="utf-8-sig")

print(f"Archivo guardado en: {ruta_salida}")

Archivo guardado en: ..\data\preprocessed\datos_2021.csv
